# Adult Income Classification
**Link:** https://archive.ics.uci.edu/dataset/2/adult

## Descripción del proyecto
Este proyecto tiene como objetivo predecir si una persona percibe ingresos anuales superiores a 50K a partir de variables sociodemográficas y laborales del dataset **Adult Census Income**.

Se trata de un problema de **clasificación binaria**, en el que la variable objetivo indica si el ingreso anual es `<=50K` o `>50K`.

## Objetivos
- Explorar el comportamiento de las variables numéricas y categóricas.
- Identificar patrones asociados a ingresos superiores a 50K.
- Construir y comparar modelos de machine learning.
- Evaluar cuáles variables aportan mayor valor predictivo.

## Tecnologías utilizadas
- Python
- Pandas, NumPy
- Scikit-learn
- Imbalanced-learn (SMOTE)
- XGBoost
- Matplotlib / Seaborn

## Metodología
- Preprocesamiento de datos (variables numéricas y categóricas)
- Manejo de desbalance de clases mediante SMOTE
- Entrenamiento con XGBoost
- Optimización de hiperparámetros con GridSearchCV
- Evaluación con métricas como ROC-AUC y F1-score

#### 1.- Librerías utilizadas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 1. Carga de datos

Se carga el dataset original y se revisa su estructura general.

#### 2.- Carga de datos

In [ ]:
# =========================================
# 2. CARGA DE DATOS
# =========================================
df = pd.read_csv("train_adultos.csv", skipinitialspace=True, sep=",") 

display(df.head())
print("Dimensión del dataset:", df.shape)


## 2. Revisión inicial

En esta sección se revisan:
- tipos de datos
- variables disponibles
- forma inicial de la variable objetivo

In [ ]:
# =========================================
# 3. REVISION INICIAL
# =========================================
display(df.dtypes.to_frame("dtype"))
print("Valores únicos de income:", df["income"].unique())


## 3. Limpieza básica

Se realiza una limpieza inicial de los datos:
- reemplazo de valores faltantes representados como `?`
- limpieza de espacios en columnas de texto
- transformación de la variable objetivo a formato numérico

In [ ]:
# =========================================
# 4. LIMPIEZA BASICA
# =========================================
df = df.replace("?", np.nan)

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].str.strip() 
#Usamos str.strip() para eliminar espacios en blanco al inicio y al final de las cadenas de texto

df["income"] = df["income"].map({"<=50K": 0, ">50K": 1}).astype(int)

display(df.head())
display(df.info())


## 4. Análisis de valores faltantes

Se analiza la presencia de nulos y su posible relación entre variables.


##### 4.1.- Valores nulos

In [ ]:
#Visualizar las variables con datos nulos
df.isna().sum()

In [ ]:
print(round((df['work-class'].isna().mean()) * 100,3))
print(round((df['occupation'].isna().mean()) * 100,3))
print(round((df['native-country'].isna().mean()) * 100,3))

Existen valores nules en las variables: Work-class, Occupatión y Native Country pero que no representán más del 6% del total de datos, igualmente analizaremos los campos nulos por si existe alguna relación entre las variables.

In [ ]:
import missingno as msno
msno.matrix(df)

Podemos ver que en los casos donde figura un valor nulo en "Work Class", también se da para "Occupation", por lo que nos dice que la persona encuestada no tiene ocupación ni trabajo actual; sin embargo, para estar seguro de ello se procede con la correlación de los nulos de las dos variables.

El campo de "Native Country" sí parece ser un campo aislado.

In [ ]:
(df[(df['work-class'].isnull()) & (df['occupation'].isnull())].shape[0] / len(df))*100

In [ ]:
df[["work-class","occupation"]].isnull().corr()

Análisis de valores nulos

- Observamos que del total de registros (32561), el 5.64% presentan valores nulos simultáneamente en 'work-class' y 'occupation'.

- Además se identificó que las variables `work-class` y `occupation` presentan valores faltantes altamente correlacionados (~0.998), lo que sugiere que la ausencia de información ocurre de manera conjunta.

Esto indica que ambos atributos comparten información similar en términos de datos faltantes.

## 5. Distribución de la variable objetivo

Se analiza la proporción de clases para verificar si existe desbalance.

In [ ]:
# =========================================
# 6. VARIABLE OBJETIVO
# =========================================
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x="income", ax=ax)
ax.set_title("Distribución de la variable objetivo")
ax.set_xlabel("income")
ax.set_ylabel("Frecuencia")
ax.set_xticklabels(["<=50K", ">50K"])
plt.show()

target_counts = df["income"].value_counts().rename(index={0: "<=50K", 1: ">50K"})
display(target_counts.to_frame("count"))

print("Proporción de clase positiva (>50K):", round(df["income"].mean() * 100, 2), "%")

El dataset presenta un desbalance moderado: la clase `<=50K` es mayoritaria, mientras que la clase `>50K` representa aproximadamente una cuarta parte del total de registros.

Por ello, en la etapa de modelado se dará especial atención a métricas como `recall`, `F1-score` y `ROC-AUC`, y no solo a la exactitud global.

## 6. Análisis de variables numéricas

Se revisan estadísticas descriptivas, distribuciones y correlaciones entre variables numéricas.

In [ ]:
# =========================================
# 7. ANALISIS NUMERICO
# =========================================
numeric_cols = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]

df[numeric_cols + ["income"]].describe()

In [ ]:
fig=df[numeric_cols].hist(bins=40, figsize=(14, 10), color="#4C72B0", edgecolor="black")
plt.suptitle("Distribuciones de variables numéricas", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
Variables = ['age', 'capital-gain', 'capital-loss', 'hours-per-week','education-num','income']
sns.pairplot(df[Variables], height=3, diag_kind = 'kde', hue='income')

Cruzamos las variables numéricas entre ellas y adiconalmente agregamos el Target para conocer la distribución y comenzar a generar hipótesis de variables que discriminan mejor a ambos grupos (Mayor y menor sueldo a 50K).

In [ ]:
num_cols = df[Variables].corr()

plt.figure(figsize=(10,6))
sns.heatmap(num_cols, annot=True, cmap="Blues", fmt=".3f")
plt.title("Correlacion entre variables numericas")
plt.show()

print(num_cols["income"].sort_values(ascending=False))

### Hallazgos numéricos

Entre las variables numéricas, `education-num` presenta la mayor asociación lineal con la variable objetivo, seguida por `age`, `hours-per-week`, `capital-gain` y `capital-loss`.

También se observa que `capital-gain` y `capital-loss` presentan distribuciones altamente sesgadas, con gran presencia de valores cercanos a cero y algunos valores extremos.

## 7. Análisis de variables categóricas

Se revisa la distribución de las variables cualitativas y su relación con la proporción de ingresos superiores a 50K.


In [ ]:
categorical_attributes = df.select_dtypes(include=['object'])

In [ ]:
categorical_attributes.dtypes

###### 4.4.1.- Work Class (Tipo de Trabajo)

In [ ]:
#Analisis Univeriado
plt.figure(figsize=(12,6))
sns.countplot(data = categorical_attributes, x = "work-class")
df['work-class'].value_counts(normalize=True)

Insight

Se observa que la mayoría de individuos pertenece a la categoría "Private", lo que refleja la distribución real del mercado laboral.

Además, ciertas categorías como "Self-emp-not-inc" presentan mayor proporción de ingresos superiores a 50K.


In [ ]:
pd.crosstab(df['work-class'], df['income'], normalize='index')

In [ ]:
#sns.factorplot('income',data=df,hue='work-class',kind="count")
df[['work-class', 'income']].groupby(['work-class'], as_index=False).agg(['count','sum','mean'])

Los segmentos de "Work-Class" con mayor proporción de salarios mayor a 50K (1) es en "Self-emp-inc" y "Federal-gov".

###### 4.4.2.- Education (Educación)

In [ ]:
df['education'].value_counts(1)

El nivel de educación "HS-grad" cuenta con el 32% de observaciones, seguido con "Some-Collegue" con 22% aprox.

In [ ]:
df[['education', 'income']].groupby(['education'], as_index=False).agg(['count','sum','mean']) #para 

Los niveles de "Education" con mayor proporción de salarios mayor a 50K (1) son "Doctorate", "Prof-school", "Masters" y "Bachelors".

###### 4.4.3.- Marital Status

In [ ]:
df['marital-status'].value_counts(1)

El nivel de marital status "Married-civ-spouse" cuenta con el 46% de observaciones, seguido con "Never-married" con 33% aprox.

In [ ]:
df[['marital-status', 'income']].groupby(['marital-status'], as_index=False).agg(['count','sum','mean'])

Los niveles de "marital-status" con mayor proporción de salarios mayor a 50K (1) es en "Married-AF-spouse" y "Married-civ-spouse".

###### 4.4.4.- Occupation

In [ ]:
df['occupation'].value_counts(1)

Los niveles de occupation Prof-specialty, Craft-repair, Exec-managerial, Adm-clerical, Sales y Other-service tienen entre un 10 y 13% de observaciones, mientras que los restantes tienen de 6% a menos.

In [ ]:
df[['occupation', 'income']].groupby(['occupation'], as_index=False).agg(['count','sum','mean'])

Los niveles de "occupation" con mayor proporción de salarios mayor a 50K (1) es en "Exec-managerial" y "Prof-specialty".

###### 4.4.5.- Relationship

In [ ]:
df['relationship'].value_counts(1)

El nivel de relationship "Husband" cuenta con el 41% de observaciones, seguido con "Not-in-family" con 26% aprox.

In [ ]:
df[['relationship', 'income']].groupby(['relationship'], as_index=False).agg(['count','sum','mean'])

Los niveles de "relationship" con mayor proporción de salarios mayor a 50K (1) es en "Wife" y "Husband".

###### 4.4.6.- Race

In [ ]:
df['race'].value_counts(1)

El nivel de race "White" cuenta con el 85% de observaciones.

In [ ]:
df[['race', 'income']].groupby(['race'], as_index=False).agg(['count','sum','mean'])

Los niveles de "race" con mayor proporción de salarios mayor a 50K (1) es en "White" y "Asian-Pac-Islander".

###### 4.4.6.- Sex

In [ ]:
df['sex'].value_counts(1)

El género Masculino cuenta con el 67% de personas, es decir 2 de cada 3 personas son masculinos.

In [ ]:
df[['sex', 'income']].groupby(['sex'], as_index=False).agg(['count','sum','mean'])

El género Masculino tiene mayor proporción de salarios mayor a 50K (1) respecto al género femenino (30% vs 11%).

###### 4.4.7.- native-country

In [ ]:
df['native-country'].value_counts(1)

In [ ]:
df[['native-country', 'income']].groupby(['native-country'], as_index=False).agg(['count','sum','mean'])

### Hallazgos categóricos

En las variables categóricas se observan patrones relevantes:
- niveles educativos altos como `Doctorate`, `Prof-school` y `Masters` muestran mayor proporción de ingresos superiores a 50K
- ocupaciones como `Exec-managerial` y `Prof-specialty` presentan una fuerte relación con la clase positiva
- categorías como `Married-civ-spouse`, `Husband` y `Wife` también muestran una mayor proporción de ingresos altos

En el caso de `native-country`, se debe tener cautela al interpretar categorías con pocos registros, ya que sus proporciones pueden ser inestables.


## 8. Preparación de datos para el modelado

En esta etapa se toman decisiones orientadas a construir un pipeline más robusto y reproducible.

### Decisiones metodológicas
- Se elimina `fnlwgt` del modelado por tratarse de una variable de peso muestral, poco interpretable en este contexto.
- Se elimina `education`, ya que `education-num` representa la misma información en formato ordinal.
- Se agrupan los países menos frecuentes en `native-country` dentro de la categoría `Other`.
- La separación entre entrenamiento y prueba se realiza antes de aprender transformaciones.


#### PREPARACION PARA MODELADO


In [ ]:
df_model = df.drop(columns=["fnlwgt", "education"]).copy()

X = df_model.drop(columns="income")
y = df_model["income"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.30,random_state=42,stratify=y)

X_train = X_train.copy()
X_test = X_test.copy()

top_paises = X_train["native-country"].value_counts(dropna=True).head(5).index.tolist()

def agrupar_paises(series, top_values):
    return series.apply(
        lambda x: x if pd.notna(x) and x in top_values else ("Other" if pd.notna(x) else np.nan) 
    )

X_train["native-country"] = agrupar_paises(X_train["native-country"], top_paises) 
X_test["native-country"] = agrupar_paises(X_test["native-country"], top_paises)

print("Dimensión X_train:", X_train.shape)
print("Dimensión X_test :", X_test.shape)
print("\nTop países conservados:", top_paises)

display(X_train["native-country"].value_counts(dropna=False).to_frame("count"))


## 9. Preprocesamiento

Se utiliza un `ColumnTransformer` para aplicar transformaciones distintas según el tipo de variable:

- Variables numéricas:
  - imputación por mediana
  - escalado con `MinMaxScaler`

- Variables categóricas:
  - imputación por categoría más frecuente
  - codificación mediante `OneHotEncoder`

El uso de `Pipeline` evita errores manuales y ayuda a prevenir data leakage.

#### Preprocesamiento

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Variables numéricas:", numeric_features)
print("Variables categóricas:", categorical_features)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


## 10. Modelado

Se comparan tres modelos de clasificación:
- XGBoost
- Random Forest
- Decision Tree

Además, se utiliza `SMOTE` dentro del pipeline para tratar el desbalance de clases de forma correcta, únicamente sobre el conjunto de entrenamiento.


#### Modelos

In [ ]:
pipeline_xgb = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42,k_neighbors=3)),
    ("model", XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ))
])

pipeline_rf = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

pipeline_dt = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("model", DecisionTreeClassifier(
        random_state=42
    ))
])

modelos = {
    "XGBoost": pipeline_xgb,
    "Random Forest": pipeline_rf,
    "Decision Tree": pipeline_dt
}


#### Función de evaluación

In [ ]:
def evaluar_modelo(nombre, pipeline, X_train, X_test, y_train, y_test):
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1] #Obtenemos la probabilidad de la clase positiva (1) para el cálculo de ROC-AUC.

    report = classification_report(y_test, y_pred, output_dict=True)

    metrics = {
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob),
        "Precision_Clase_1": report["1"]["precision"],
        "Recall_Clase_1": report["1"]["recall"],
        "F1_Clase_1": report["1"]["f1-score"]
    }

    print(f"\n{'='*70}")
    print(f"MODELO: {nombre}")
    print(f"Accuracy: {metrics['Accuracy']:.4f}")
    print(f"ROC-AUC : {metrics['ROC_AUC']:.4f}")
    print("\nClassification report:")
    print(classification_report(y_test, y_pred))

    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
    plt.title(f"Matriz de confusión - {nombre}")
    plt.show()

    return metrics


#### Comparación de modelos

In [ ]:
resultados = []

for nombre, modelo in modelos.items():
    resultados.append(
        evaluar_modelo(nombre, modelo, X_train, X_test, y_train, y_test)
    )

df_resultados = pd.DataFrame(resultados).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
display(df_resultados)


## 11. Ajuste de hiperparámetros de XGBoost

Una vez identificado XGBoost como el modelo más competitivo, se realiza una búsqueda de hiperparámetros con `GridSearchCV`.

La métrica de optimización elegida es `F1-score`, ya que en este problema interesa equilibrar precisión y recall de la clase positiva.


##### TUNING DE XGBOOST

In [ ]:
param_grid = {
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.1],
    "model__n_estimators": [100, 200]
}

grid_xgb = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=param_grid,
    scoring="f1",
    cv = 3,
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

print("Mejores parámetros:")
print(grid_xgb.best_params_)
print("\nMejor F1 en validación cruzada:")
print(round(grid_xgb.best_score_, 4))


##### Comparación XGB Base vs. Tuneado:

In [ ]:
base_xgb = pipeline_xgb
tuned_xgb = grid_xgb.best_estimator_ 

comparacion_xgb = []

for nombre, modelo in {
    "XGBoost Base": base_xgb,
    "XGBoost Tuneado": tuned_xgb
}.items():
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    report = classification_report(y_test, y_pred, output_dict=True)

    comparacion_xgb.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_prob),
        "Precision_Clase_1": report["1"]["precision"],
        "Recall_Clase_1": report["1"]["recall"],
        "F1_Clase_1": report["1"]["f1-score"]
    })

df_xgb_compare = pd.DataFrame(comparacion_xgb).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
display(df_xgb_compare)


## 12. Selección del modelo final

La comparación anterior permite decidir si se conserva el modelo base o el modelo tuneado.

En muchos casos, el modelo tuneado puede mejorar el `recall` de la clase positiva sin necesariamente mejorar el `ROC-AUC` o la exactitud global. Por ello, la selección final debe interpretarse como un trade-off entre desempeño general y capacidad de detección.

#### Selección del modelo final

In [ ]:
base_row = df_xgb_compare[df_xgb_compare["Modelo"] == "XGBoost Base"].iloc[0]
tuned_row = df_xgb_compare[df_xgb_compare["Modelo"] == "XGBoost Tuneado"].iloc[0]

if tuned_row["ROC_AUC"] > base_row["ROC_AUC"]:
    final_model = tuned_xgb
    final_model_name = "XGBoost Tuneado"
    final_metrics = tuned_row
else:
    final_model = base_xgb
    final_model_name = "XGBoost Base"
    final_metrics = base_row

print("Modelo final seleccionado:", final_model_name)
display(final_metrics.to_frame("Valor"))


## 13. Importancia de variables

Se analiza la importancia de variables del modelo final para identificar qué atributos aportan mayor capacidad predictiva.


In [ ]:
feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
importances = final_model.named_steps["model"].feature_importances_

df_importancia = (
    pd.DataFrame({
        "Variable": feature_names,
        "Importancia": importances
    })
    .sort_values("Importancia", ascending=False)
    .reset_index(drop=True)
)

display(df_importancia.head(15))


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_importancia.head(15),
    x="Importancia",
    y="Variable",
    hue="Variable",
    dodge=False,
    palette="viridis",
    legend=False
)
plt.title("Top 15 variables más importantes")
plt.show()


## 14. Conclusiones

El objetivo del proyecto fue predecir si una persona percibe ingresos anuales superiores a 50K utilizando variables sociodemográficas y laborales del dataset **Adult Census Income**. A nivel de desempeño predictivo, **XGBoost** fue el modelo con mejor resultado general frente a **Random Forest** y **Decision Tree**, mostrando el mejor equilibrio entre capacidad discriminativa y desempeño sobre la clase positiva.

Desde el análisis exploratorio, se observó que las variables numéricas con mayor asociación con la variable objetivo fueron **education-num**, **age**, **hours-per-week**, **capital-gain** y **capital-loss**. En particular, **education-num** presentó la correlación más alta con `income`, lo que sugiere que el nivel educativo tiene un papel especialmente relevante en la predicción del ingreso.

En las variables categóricas también aparecieron patrones claros. En educación, categorías como **Doctorate**, **Prof-school** y **Masters** concentraron las mayores proporciones de ingresos superiores a 50K. Esto podría estar relacionado con que niveles educativos más altos suelen facilitar el acceso a ocupaciones de mayor especialización y mejor remuneración, aunque este análisis debe interpretarse en términos predictivos y no causales.

En ocupación, categorías como **Exec-managerial** y **Prof-specialty** mostraron una fuerte relación con la clase positiva, lo que resulta coherente con el tipo de cargos asociados a mayor responsabilidad, especialización o nivel salarial. De forma similar, variables vinculadas a la estructura familiar, como **Married-civ-spouse**, **Husband** y **Wife**, también presentaron mayores proporciones de ingresos altos, lo que sugiere que la situación familiar y laboral puede estar asociada con distintos perfiles de ingreso en el dataset.

El análisis de importancia de variables confirmó que no solo el nivel educativo aporta valor predictivo, sino también variables relacionadas con estado civil, ocupación, relación familiar y ganancias de capital. Esto refuerza la idea de que el ingreso no depende de un único factor, sino de la combinación de características educativas, laborales y demográficas.

Sin embargo, el trabajo presenta algunas limitaciones. El dataset tiene desbalance de clases, algunas categorías cuentan con pocos registros y variables sensibles como sexo, raza y país de origen requieren especial cautela por el posible riesgo de sesgo. Además, los resultados obtenidos deben interpretarse como asociaciones útiles para predicción, no como evidencia de relaciones causales.

Como líneas futuras de trabajo, sería recomendable ajustar el umbral de clasificación según el objetivo del problema, evaluar métricas de fairness entre grupos, comparar con modelos más interpretables y validar el modelo con datos externos o con esquemas de validación más robustos.

